# Statcast data fetch

Pulls MLB regular-season Statcast data to `./data/<year>_data.csv`, one file
per season, for the swing-decision model.

Each season is filtered on two things:

1. `game_type == 'R'` — regular season only.
2. **Games outside the US and Canada are dropped.** International series are
   played at neutral sites where the tracking system is a temporary
   installation, so calibration may differ from a regular park. This removes
   Mexico City (2023, 2024, 2026), London (2023, 2024), Seoul (2024) and Tokyo
   (2025) — 14 games across 2021–2026. Toronto is **kept**: Rogers Centre is a
   permanent park with a permanent Hawk-Eye rig.

The pitch-level Statcast export has no venue column, and international series
keep an MLB club as `home_team`, so neither column identifies these games. The
filter resolves `game_pk` to a venue through the MLB Stats API instead.

Season date ranges also come from the API rather than being hardcoded, so the
window is the actual regular season for each year.

In [ ]:
import datetime as dt
from pathlib import Path

import pandas as pd
import requests
from pybaseball import statcast

STATS_API = 'https://statsapi.mlb.com/api/v1'
DATA_DIR = Path('./data')

# Rogers Centre is a permanent MLB park with a permanent Hawk-Eye installation,
# so Canada stays. What is excluded is the overseas neutral-site series --
# Mexico City, London, Seoul, Tokyo -- where tracking is a temporary rig and
# calibration may differ from a regular park.
KEEP_COUNTRIES = {'USA', 'Canada'}


def season_bounds(year):
    """Regular-season start/end dates from the MLB Stats API."""
    r = requests.get(f'{STATS_API}/seasons', params={'sportId': 1, 'season': year}, timeout=30)
    r.raise_for_status()
    s = r.json()['seasons'][0]
    return s['regularSeasonStartDate'], s['regularSeasonEndDate']


def game_venues(year):
    """game_pk -> venue / city / country for every regular-season game in `year`."""
    start, end = season_bounds(year)
    r = requests.get(f'{STATS_API}/schedule',
                     params={'sportId': 1, 'gameType': 'R', 'startDate': start,
                             'endDate': end, 'hydrate': 'venue(location)'}, timeout=120)
    r.raise_for_status()
    rows = [{'game_pk': g['gamePk'],
             'venue': g['venue']['name'],
             'city': g['venue'].get('location', {}).get('city'),
             'country': g['venue'].get('location', {}).get('country')}
            for date in r.json()['dates'] for g in date['games']]
    return pd.DataFrame(rows)


def drop_overseas(df, year):
    """Drop rows from games played outside the US/Canada. Reports what it removed."""
    overseas = game_venues(year).query('country not in @KEEP_COUNTRIES')
    if overseas.empty:
        print(f'  no overseas games')
        return df

    pk = pd.to_numeric(df['game_pk'], errors='coerce').astype('Int64')
    for venue, grp in overseas.groupby('venue'):
        n = pk.isin(set(grp['game_pk'])).sum()
        loc = grp.iloc[0]
        print(f'  dropping {venue}, {loc.city} ({loc.country}) - {n:,} pitches')
    return df[~pk.isin(set(overseas['game_pk']))]


def fetch_season(year):
    """Pull one regular season: game_type 'R' only, overseas games removed, written to CSV.

    Returns the output path rather than the frame -- a season is ~700k rows by
    ~119 columns, so holding several at once is not worth the memory.
    """
    start, end = season_bounds(year)
    today = dt.date.today().isoformat()
    if end > today:
        end = today
        print(f'{year}: season still in progress - pulling through {end}; re-run after the finale')

    df = statcast(start, end)
    n_raw = len(df)
    df = df[df['game_type'] == 'R']
    df = drop_overseas(df, year)

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    out = DATA_DIR / f'{year}_data.csv'
    df.to_csv(out, index=False)
    print(f'{year}: {n_raw:,} pitches pulled -> {len(df):,} kept -> {out}\n')
    return out

In [ ]:
# Edit YEARS to pull only what you need -- each season is a slow download.
YEARS = range(2021, 2027)

for year in YEARS:
    fetch_season(year)

## Notes

- **2026 is incomplete** until the regular-season finale (2026-09-27). Running
  the cell above before then pulls through today and says so; re-run afterwards
  to top it up.
- Location features are **not** comparable across the 2025/2026 boundary
  without harmonization: `plate_x`/`plate_z` moved from front-of-plate to
  middle-of-plate in 2026, and `sz_top`/`sz_bot` switched to the ABS-defined
  zone. See `IMPROVEMENT_PLAN.md` §0.1.1 — that conversion belongs in the
  cleaning step, not here; this notebook writes raw pulls.
- `data/` is gitignored, so these files stay local.